# ResNet18 Edge Deployment with Comprexx

This notebook walks through compressing a ResNet18 for edge deployment:

1. Profile the original model
2. Fuse Conv+BN layers (removes BatchNorm, zero accuracy cost)
3. Prune 40% of filters by L1 norm
4. Benchmark before/after
5. Export to ONNX

Install: `pip install "comprexx[onnx]" torchvision`

In [1]:
import os

import torch
from torchvision.models import resnet18

import comprexx as cx

## 1. Load and profile the model

In [2]:
model = resnet18(weights=None)
model.eval()

input_shape = (1, 3, 224, 224)
profile = cx.analyze(model, input_shape)
print(profile.summary())

Model: ResNet
  Architecture: cnn
  Parameters:   11,689,512 (11.7M)
  Trainable:    11,689,512
  FLOPs:        3,638,082,536 (3.64 GFLOPs)
  Size:         44.63 MB
  Layers:       52 (21 compressible)


## 2. Compress: fuse + prune

Two stages:
- **Operator fusion**: folds each Conv2d + BatchNorm2d pair into a single Conv2d. This eliminates the BatchNorm layers entirely with zero accuracy cost.
- **Structured pruning**: ranks all conv filters globally by L1 norm and zeros out the bottom 40%. The pruned filters remain in the model (mask-based pruning), so the parameter count stays the same but 40% of the weights are zero.

In [3]:
pipeline = cx.Pipeline([
    cx.stages.OperatorFusion(),
    cx.stages.StructuredPruning(sparsity=0.4, criteria="l1_norm", scope="global"),
])

result = pipeline.run(model, input_shape=input_shape)
print(result.report.summary())

Compression Report: ResNet
  Total duration: 0.17s
  Stages:         2
  Compression:    1.00x
  Size reduction: 0.1%
  FLOPs reduction:0.2%

Stage: operator_fusion (operator_fusion_fx)
  Duration:    0.07s
  Params:      11,689,512 -> 11,684,712
  Size:        46.80 MB -> 46.74 MB (0.1% reduction)
  FLOPs:       3,638,082,536 -> 3,630,631,400 (0.2% reduction)
  Notes:       Fused 20 Conv2d+BatchNorm2d pair(s).

Stage: structured_pruning (structured_pruning_l1_norm)
  Duration:    0.08s
  Params:      11,684,712 -> 11,684,712
  Size:        46.74 MB -> 46.74 MB (0.0% reduction)
  FLOPs:       3,630,631,400 -> 3,630,631,400 (0.0% reduction)
  Notes:       Global pruning: zeroed 1920 filters across 14 layers.



## 3. Verify the compressed model produces valid output

In [4]:
compressed = result.model

with torch.no_grad():
    x = torch.randn(*input_shape)
    out = compressed(x)

print(f"Output shape: {out.shape}")
print(f"Output range: [{out.min():.3f}, {out.max():.3f}]")

Output shape: torch.Size([1, 1000])
Output range: [-0.044, 0.044]


## 4. Benchmark inference latency

Mask-based structured pruning zeroes weights but does not physically remove filters, so latency stays roughly the same. The benefit is in model size after serialization (zeros compress well) and as a precursor to hardware-accelerated sparse inference.

In [5]:
comparison = cx.compare_benchmarks(
    model, compressed,
    input_shape=input_shape,
    warmup=10,
    iters=50,
)
print(comparison.summary())

Benchmark Comparison (cpu)
  Baseline:   7.938 ms  (126.0 ips)
  Compressed: 7.714 ms  (129.6 ips)
  Speedup:    1.03x  (+2.8% latency, +2.9% throughput)


## 5. Export to ONNX

The exporter runs `torch.onnx.export`, validates the output against PyTorch, and writes a `comprexx_manifest.json` with compression metadata.

In [6]:
exporter = cx.ONNXExporter()
exporter.export(
    compressed,
    input_shape=input_shape,
    output_path="resnet18_compressed.onnx",
)

onnx_size = os.path.getsize("resnet18_compressed.onnx") / 1e6
print(f"ONNX file size: {onnx_size:.1f} MB")

[torch.onnx] Obtain model graph for `GraphModule([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `GraphModule([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
Applied 20 of general pattern rewrite rules.
[torch.onnx] Optimize the ONNX graph... ✅
ONNX file size: 0.1 MB


## 6. Same pipeline as a YAML recipe

You can define this exact pipeline in YAML and run it from the CLI.

In [7]:
recipe_yaml = """
name: resnet18-edge
description: Fused and pruned ResNet18 for edge deployment

stages:
  - technique: operator_fusion

  - technique: structured_pruning
    sparsity: 0.4
    criteria: l1_norm
    scope: global
"""

print(recipe_yaml)
print("# Save as resnet18-edge.yaml, then run:")
print(
    "# comprexx compress torchvision.models.resnet18"
    " --recipe resnet18-edge.yaml --input-shape 1,3,224,224"
)


name: resnet18-edge
description: Fused and pruned ResNet18 for edge deployment

stages:
  - technique: operator_fusion

  - technique: structured_pruning
    sparsity: 0.4
    criteria: l1_norm
    scope: global

# Save as resnet18-edge.yaml, then run:
# comprexx compress torchvision.models.resnet18 --recipe resnet18-edge.yaml --input-shape 1,3,224,224
